# Team Basketball Analytics - PELISSANNE BASKET AVENIR

Analyse complète avec évolution des ratings Elo et projection de fin de saison

In [ ]:
# Importations
import pandas as pd
from ffbb_api_client_v2 import FFBBAPIClientV2, TokenManager
from ffbb_api_client_v2.models.type_competition import TypeCompetition

# Configuration
TEAM_NAME = "PELISSANNE BASKET AVENIR"
COMPETITION_FILTERS = {
    "sexe": "M",
    "type_competition": TypeCompetition.CHAMPIONNAT,
}

# Initialisation API
tokens = TokenManager.get_tokens()
client = FFBBAPIClientV2.create(
    meilisearch_bearer_token=tokens.meilisearch_token,
    api_bearer_token=tokens.api_token,
    debug=False,
)
print("✅ FFBB API Client initialized successfully!")

In [ ]:
# Fonction pour trouver l'ID de poule
# Note: organisme.engagements is list[int] (FK IDs), NOT engagement objects.

def find_team_poule_id(client, team_name, filters):
    search_results = client.search_organismes(name=team_name)
    if not search_results or not search_results.hits:
        raise ValueError(f"Team {team_name} not found")

    hit_id = search_results.hits[0].id
    if not hit_id:
        raise ValueError(f"Team {team_name} has no ID")
    organisme_id = int(hit_id)
    organisme = client.get_organisme(organisme_id)
    if not organisme:
        raise ValueError(f"Organisme {organisme_id} not found")

    # organisme.engagements is list[int] — resolve via batch fetch
    engagement_ids = [eid for eid in organisme.engagements if isinstance(eid, int)]
    if not engagement_ids:
        raise ValueError(f"No engagements found for {team_name}")

    engagements = client.list_engagements_by_ids(engagement_ids)

    for eng in engagements:
        if not eng.idCompetition or not eng.idPoule:
            continue
        comp = client.get_competition(eng.idCompetition)
        if not comp:
            continue
        if (comp.sexe == filters["sexe"]
            and comp.type_competition == filters["type_competition"]):
            return eng.idPoule

    raise ValueError(f"No poule found for {team_name}")

# Trouver l'ID de la poule
poule_id = find_team_poule_id(client, TEAM_NAME, COMPETITION_FILTERS)
poule_data = client.get_poule(poule_id)
if not poule_data:
    raise ValueError(f"Poule {poule_id} not found")

# poule_data.rencontres is list[int] (FK IDs) — fetch actual rencontre objects
rencontres = client.list_rencontres_by_poule(poule_id)

# Extraire les données de classement
rankings_data = []
for ranking in (poule_data.classements or []):
    if not ranking.id_engagement:
        continue
    rankings_data.append(
        {
            "position": ranking.position,
            "team_name": ranking.id_engagement.nom,
            "points": ranking.points,
            "wins": ranking.gagnes,
            "losses": ranking.perdus,
            "games_played": ranking.match_joues,
            "points_scored": ranking.paniers_marques,
            "points_allowed": ranking.paniers_encaisses,
            "point_diff": ranking.difference,
        }
    )

rankings_df = pd.DataFrame(rankings_data)
rankings_df.set_index("position", inplace=True)
print(f"✅ Data loaded for {len(rankings_df)} teams, {len(rencontres)} rencontres")

In [ ]:
# Système de rating Elo
def calculate_expected_score(rating_a, rating_b):
    """Calculate expected score for team A vs team B using Elo formula"""
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def calculate_rating_evolution(classements, rencontres):
    """Calculate rating evolution for all teams throughout the season."""
    # Initialize ratings
    team_ratings = {}
    rating_history = {}
    
    for ranking in (classements or []):
        if not ranking.id_engagement:
            continue
        team_name = ranking.id_engagement.nom
        team_ratings[team_name] = 1500.0
        rating_history[team_name] = []
    
    # rencontres are GetRencontresResponse objects (fetched via list_rencontres_by_poule)
    played_matches = [r for r in rencontres if r.joue]
    played_matches.sort(key=lambda x: x.date_rencontre or x.date or x.id)
    
    K = 32
    
    for match_idx, match in enumerate(played_matches):
        team1 = match.nomEquipe1
        team2 = match.nomEquipe2
        
        if not team1 or not team2:
            continue
        if team1 not in team_ratings or team2 not in team_ratings:
            continue
        
        score1 = match.resultatEquipe1 or 0
        score2 = match.resultatEquipe2 or 0
        
        if score1 > score2:
            actual1, actual2 = 1, 0
        elif score1 < score2:
            actual1, actual2 = 0, 1
        else:
            actual1, actual2 = 0.5, 0.5
        
        rating1 = team_ratings[team1]
        rating2 = team_ratings[team2]
        
        expected1 = calculate_expected_score(rating1, rating2)
        expected2 = calculate_expected_score(rating2, rating1)
        
        new_rating1 = rating1 + K * (actual1 - expected1)
        new_rating2 = rating2 + K * (actual2 - expected2)
        
        team_ratings[team1] = new_rating1
        team_ratings[team2] = new_rating2
        
        for team in team_ratings:
            rating_history[team].append({
                'match': match_idx + 1,
                'rating': team_ratings[team],
                'change': 0.0
            })
        
        # Calculate changes
        for team in rating_history:
            if len(rating_history[team]) > 1:
                rating_history[team][-1]['change'] = (
                    rating_history[team][-1]['rating'] - rating_history[team][-2]['rating']
                )
            else:
                rating_history[team][-1]['change'] = rating_history[team][-1]['rating'] - 1500
    
    return rating_history, team_ratings

# Calculate ratings using fetched rencontres
rating_history, final_ratings = calculate_rating_evolution(poule_data.classements, rencontres)
print(f"✅ Rating evolution calculated for {len(rating_history)} teams")
print(f"Matches processed: {len(next(iter(rating_history.values())))}")

In [4]:
# Convert to DataFrames
rating_dfs = {}
for team, history in rating_history.items():
    rating_dfs[team] = pd.DataFrame(history)

# Final ratings ranking with normalized ratings
final_rating_df = pd.DataFrame({
    'team_name': list(final_ratings.keys()),
    'final_rating': list(final_ratings.values())
}).sort_values('final_rating', ascending=False).reset_index(drop=True)

final_rating_df['rank'] = final_rating_df.index + 1
final_rating_df['rating_change'] = final_rating_df['final_rating'] - 1500
final_rating_df['normalized_rating'] = (final_rating_df['final_rating'] - final_rating_df['final_rating'].min()) / (final_rating_df['final_rating'].max() - final_rating_df['final_rating'].min())

print("🏆 Final Elo Rating Rankings")
print("=" * 40)
for _, row in final_rating_df.head(5).iterrows():
    change_symbol = "+" if row['rating_change'] >= 0 else ""
    print(f"{int(row['rank']):2d}. {row['team_name'][:20]:20} {row['final_rating']:6.1f} ({change_symbol}{row['rating_change']:5.1f}) | Norm: {row['normalized_rating']:.3f}")

print("...")

# Show target team
target_row = final_rating_df[final_rating_df['team_name'] == TEAM_NAME]
if not target_row.empty:
    row = target_row.iloc[0]
    change_symbol = "+" if row['rating_change'] >= 0 else ""
    print(f"{int(row['rank']):2d}. {row['team_name'][:20]:20} {row['final_rating']:6.1f} ({change_symbol}{row['rating_change']:5.1f}) | Norm: {row['normalized_rating']:.3f}")

🏆 Final Elo Rating Rankings
 1. ELAN BASKET PERNOIS  1647.0 (+147.0) | Norm: 1.000
 2. UNION SPORTIVE AVIGN 1644.1 (+144.1) | Norm: 0.990
 3. OLYMPIQUE CARROS BAS 1558.7 (+ 58.7) | Norm: 0.689
 4. ISTRES SPORTS BC     1541.7 (+ 41.7) | Norm: 0.629
 5. US CAGNES SUR MER    1500.0 (+  0.0) | Norm: 0.482
...
11. PELISSANNE BASKET AV 1370.2 (-129.8) | Norm: 0.025


In [ ]:
# Projection de fin de saison
# Uses `rencontres` (list of GetRencontresResponse) instead of poule_data.rencontres (list[int])

def project_season_end(rencontres, current_ratings, team_name):
    """Project season end assuming wins against similar-rated opponents"""
    
    projected_ratings = current_ratings.copy()
    remaining_matches = [r for r in rencontres if not r.joue]
    target_remaining = [m for m in remaining_matches if m.nomEquipe1 == team_name or m.nomEquipe2 == team_name]
    
    K = 32
    simulated_wins = 0
    
    for match in target_remaining:
        if match.nomEquipe1 == team_name:
            opponent = match.nomEquipe2
        else:
            opponent = match.nomEquipe1
        
        if not opponent or opponent not in projected_ratings:
            continue
        
        rating_diff = abs(projected_ratings[team_name] - projected_ratings[opponent])
        
        if rating_diff <= 100:
            simulated_wins += 1
            rating_team = projected_ratings[team_name]
            rating_opp = projected_ratings[opponent]
            
            expected_team = calculate_expected_score(rating_team, rating_opp)
            expected_opp = calculate_expected_score(rating_opp, rating_team)
            
            new_rating_team = rating_team + K * (1 - expected_team)
            new_rating_opp = rating_opp + K * (0 - expected_opp)
            
            projected_ratings[team_name] = new_rating_team
            projected_ratings[opponent] = new_rating_opp
    
    return projected_ratings, simulated_wins

# Project season end using fetched rencontres
projected_ratings, simulated_wins = project_season_end(rencontres, final_ratings, TEAM_NAME)

# Calculate projected normalized rating
projected_rating_df = pd.DataFrame({
    'team_name': list(projected_ratings.keys()),
    'projected_rating': list(projected_ratings.values())
}).sort_values('projected_rating', ascending=False).reset_index(drop=True)

projected_rating_df['projected_normalized'] = (projected_rating_df['projected_rating'] - projected_rating_df['projected_rating'].min()) / (projected_rating_df['projected_rating'].max() - projected_rating_df['projected_rating'].min())
projected_rating_df['rank'] = projected_rating_df.index + 1
projected_rating_df['change_from_current'] = projected_rating_df['projected_rating'] - final_rating_df.set_index('team_name')['final_rating']

# Show projection for target team
target_projected = projected_rating_df[projected_rating_df['team_name'] == TEAM_NAME]

print("🎯 PROJECTION DE FIN DE SAISON")
print("=" * 50)
print(f"Équipe: {TEAM_NAME}")
print(f"Matches simulés gagnés: {simulated_wins}")
print(f"Rating actuel: {final_ratings[TEAM_NAME]:.1f}")
print(f"Rating projeté: {projected_ratings[TEAM_NAME]:.1f} (+{projected_ratings[TEAM_NAME] - final_ratings[TEAM_NAME]:.1f})")
print(f"Rang projeté: {target_projected['rank'].iloc[0]}/{len(projected_rating_df)}")
print(f"Rating normalisé projeté: {target_projected['projected_normalized'].iloc[0]:.3f}")

print("\n🏆 Classement Projeté Top 5:")
for _, row in projected_rating_df.head(5).iterrows():
    change = row['change_from_current']
    change_str = f"+{change:5.1f}" if not pd.isna(change) else "  N/A"
    print(f"{int(row['rank']):2d}. {row['team_name'][:20]:20} {row['projected_rating']:6.1f} ({change_str}) | Norm: {row['projected_normalized']:.3f}")